# 공통 전처리

호텔 예약 취소 예측 · 2026-10-29 팀 회의 결정 사항 · 이 파일 하나로 전처리가 끝납니다.

- 예측 시점: **예약이 확정된 직후** (이 시점에 알 수 없는 정보는 사용하지 않음)
- 입력: `hotel_bookings.csv` (노트북과 같은 폴더 또는 `../data/raw/`)
- 출력: `hotel_preprocessed.csv` (입력 파일과 같은 위치의 processed 폴더, 없으면 노트북 폴더)

| 단계 | 결정 |
| --- | --- |
| 결측치 | children 최빈값(0), country "Unknown", company → has_company 0·1, agent 범주(결측 "Unknown") |
| 누수 컬럼 | reservation_status, reservation_status_date, assigned_room_type, booking_changes, required_car_parking_spaces 제거 / days_in_waiting_list 유지 |
| 이상 행 | 투숙객 0명, 음수 adr, adr 5000 이상 제거 / 0박 예약 유지 |
| 파생변수 | total_guests, total_nights (재료 컬럼도 유지) |
| 중복 행 | 삭제하지 않고 group_id로 묶어 그룹 분할 |
| 분할 | StratifiedGroupKFold, train 80 / test 20, random_state=0 → `split` 컬럼 |
| 인코딩·스케일링 | CSV에는 하지 않음 → 각자 Pipeline에서 train만 fit |

## 0. 설정

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 0   # 팀 공통 고정값
N_SPLITS = 5       # 5개로 나눈 것 중 1개를 test로 → 약 80 / 20
TARGET = "is_canceled"

# 원본 위치: 노트북과 같은 폴더 → 없으면 ../data/raw/
candidates = [Path("hotel_bookings.csv"), Path("../data/raw/hotel_bookings.csv")]
RAW_PATH = next((p for p in candidates if p.exists()), None)
assert RAW_PATH is not None, "hotel_bookings.csv를 노트북과 같은 폴더나 ../data/raw/에 넣어 주세요."

# 결과 위치: ../data/processed/ 가 있으면 거기, 없으면 노트북 폴더
OUT_DIR = Path("../data/processed") if Path("../data").exists() else Path(".")
OUT_PATH = OUT_DIR / "hotel_preprocessed.csv"

# 예약 확정 직후에 알 수 없는 컬럼 (누수)
LEAK_COLS = [
    "reservation_status",           # 예약의 최종 상태 (결과 그 자체)
    "reservation_status_date",      # 최종 상태가 바뀐 날짜 (취소일)
    "assigned_room_type",           # 체크인할 때 배정되는 방
    "booking_changes",              # 확정 후 체크인 전까지 쌓이는 변경 횟수
    "required_car_parking_spaces",  # 요청 예약 취소율 0% → 체크인 때 기록된 값으로 의심
]

ID_COLS = ["source_row_id", "group_id", "split"]   # 모델 입력이 아닌 관리용 컬럼

CAT_COLS = [   # 범주형 (원-핫 대상)
    "hotel", "arrival_date_month", "meal", "country", "market_segment",
    "distribution_channel", "reserved_room_type", "deposit_type",
    "agent", "customer_type",
]
NUM_COLS = [   # 수치형 (로지스틱·MLP는 스케일링 대상)
    "lead_time", "arrival_date_year", "arrival_date_week_number",
    "arrival_date_day_of_month", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
    "days_in_waiting_list", "adr", "total_of_special_requests",
    "has_company", "total_guests", "total_nights",
]

print("원본:", RAW_PATH, "| 결과:", OUT_PATH)

원본: hotel_bookings.csv | 결과: hotel_preprocessed.csv


## 1. 불러오기

In [2]:
df = pd.read_csv(RAW_PATH)
df.insert(0, "source_row_id", df.index)   # 원본 행 번호 (추적용)

print("원본:", df.shape, "| 취소율:", round(df[TARGET].mean(), 4))
df.isna().sum()[lambda s: s > 0]

원본: (119390, 33) | 취소율: 0.3704


children         4
country        488
agent        16340
company     112593
dtype: int64

## 2. 결측치 처리

| 컬럼 | 처리 | 이유 |
| --- | --- | --- |
| children (4건) | 최빈값 0 | 4건뿐이라 결과 차이 거의 없음 |
| country (488건) | "Unknown" | 모르는 것도 하나의 정보 |
| company (94% 결측) | has_company 0·1 후 삭제 | ID 352종류, 회사 예약 여부만 남김 |
| agent (16,340건) | 글자(범주)로, 결측 "Unknown" | 크기 의미가 없는 ID. 범주가 0·1보다 AUC 높았음 |

In [3]:
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")

df["has_company"] = df["company"].notna().astype(int)
df = df.drop(columns=["company"])

# 9.0 → "9" 처럼 정수 글자로
df["agent"] = df["agent"].apply(lambda v: "Unknown" if pd.isna(v) else str(int(v)))

print("남은 결측:", df.isna().sum().sum())
print("agent 종류:", df["agent"].nunique(), "| Unknown:", (df["agent"] == "Unknown").sum())

남은 결측: 0
agent 종류: 334 | Unknown: 16340


## 3. 누수 컬럼 제거

In [4]:
before = df.shape[1]
df = df.drop(columns=LEAK_COLS)
print("컬럼 수:", before, "→", df.shape[1])

컬럼 수: 33 → 28


## 4. 이상 행 제거

| 조건 | 처리 | 이유 |
| --- | --- | --- |
| 투숙객 0명 | 제거 | 묵을 사람이 없는 잘못된 기록 |
| 음수 adr | 제거 | 1박 요금이 마이너스일 수 없음 |
| adr 5000 이상 | 제거 | 5,400 한 건만 혼자 튀는 값 |
| 0박 예약 | 유지 | 당일 이용 같은 실제 예약일 수 있음 |

In [5]:
guests = df["adults"] + df["children"] + df["babies"]
keep = (guests > 0) & (df["adr"] >= 0) & (df["adr"] < 5000)

before = len(df)
df = df[keep].copy()
print("제거:", before - len(df), "| 남은 행:", len(df))

제거: 182 | 남은 행: 119208


## 5. 파생변수

In [6]:
df["total_guests"] = (df["adults"] + df["children"] + df["babies"]).astype(int)   # 총 투숙객
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]    # 총 숙박일

df[["adults", "children", "babies", "total_guests",
    "stays_in_weekend_nights", "stays_in_week_nights", "total_nights"]].head()

,adults,children,babies,total_guests,stays_in_weekend_nights,stays_in_week_nights,total_nights
0,2,0.0,0,2,0,0,0
1,2,0.0,0,2,0,0,0
2,1,0.0,0,1,0,1,1
3,1,0.0,0,1,0,1,1
4,2,0.0,0,2,0,2,2


## 6. 그룹 만들기 (중복 행 처리)

입력 컬럼이 모두 같은 행에 같은 `group_id`를 붙입니다. 중복 행은 단체·여행사의 실제 예약으로 보고 지우지 않지만,
같은 묶음의 99%가 함께 취소되거나 함께 묵어서 train과 test에 나뉘면 정답을 미리 보는 누수가 생깁니다.

In [7]:
feature_cols = [c for c in df.columns if c not in ["source_row_id", TARGET]]
row_hash = pd.util.hash_pandas_object(df[feature_cols], index=False)
df["group_id"] = pd.factorize(row_hash)[0]   # 0, 1, 2 … 짧은 번호로

n_groups = df["group_id"].nunique()
print("행:", len(df), "| 그룹:", n_groups, "| 다른 행과 겹치는 행:", len(df) - n_groups)

행: 119208 | 그룹: 84668 | 다른 행과 겹치는 행: 34540


## 7. train / test 분할

`StratifiedGroupKFold(5)`의 첫 번째 묶음을 test로 씁니다 (약 80 / 20).
취소 비율은 맞추고, 같은 그룹은 한쪽에만 들어갑니다.

In [8]:
df = df.reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
_, test_idx = next(sgkf.split(df, df[TARGET], groups=df["group_id"]))

df["split"] = "train"
df.loc[test_idx, "split"] = "test"

summary = df.groupby("split").agg(행수=(TARGET, "size"), 취소율=(TARGET, "mean"))
summary["비율"] = summary["행수"] / len(df)
summary.round(4)

,행수,취소율,비율
split,,,
test,23841,0.3707,0.2
train,95367,0.3708,0.8


## 8. 검증

In [9]:
train_groups = set(df.loc[df["split"] == "train", "group_id"])
test_groups = set(df.loc[df["split"] == "test", "group_id"])

assert len(train_groups & test_groups) == 0, "train과 test에 같은 그룹이 있어요!"
assert df.isna().sum().sum() == 0, "결측이 남아 있어요!"
print("train / test 그룹 겹침: 0건")
print("결측: 0건")

train / test 그룹 겹침: 0건
결측: 0건


## 9. 저장

In [10]:
df = df[ID_COLS + CAT_COLS + NUM_COLS + [TARGET]]   # 관리용 → 입력 → 정답 순서

OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("저장:", OUT_PATH, df.shape)

저장: hotel_preprocessed.csv (119208, 32)


## 10. 모델 담당자 사용법

이 노트북을 실행하지 않아도, 결과 CSV만 받으면 아래 코드로 바로 시작할 수 있습니다.

```python
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

df = pd.read_csv("hotel_preprocessed.csv", dtype={"agent": str})

ID_COLS = ["source_row_id", "group_id", "split"]
TARGET = "is_canceled"
FEATURES = [c for c in df.columns if c not in ID_COLS + [TARGET]]

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
X_train, y_train, groups_train = train[FEATURES], train[TARGET], train["group_id"]
X_test, y_test = test[FEATURES], test[TARGET]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

# 교차검증·튜닝할 때 groups를 꼭 넣기
cross_validate(model, X_train, y_train, cv=cv, groups=groups_train)
search.fit(X_train, y_train, groups=groups_train)
```

- 범주형 10개는 원-핫 권장: `OneHotEncoder(handle_unknown="ignore", min_frequency=100)`
- 스케일링은 로지스틱·MLP만 Pipeline 안에서
- test는 최종 후보를 정한 뒤 한 번만 평가